# [Super AI Engineer Season 6] Hackathon Week 2
## การแข่งขัน OCR เอกสารผลเลือกตั้ง สส. 2569

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: เอกสารผลเลือกตั้ง สส. 2569
- Source: กกต. (Election Commission of Thailand)  
- จัดทำโดย: **600425-วิศิษฐ์**

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. OCR Pipeline Setup  
4. Batch OCR Processing  
5. Data Cleaning & Preparation  
6. Submission Generation  
7. Insight Summary & Recommendations

# 1. Setup & Imports
### 1.1 ติดตั้งและนำเข้าไลบรารี (Setup Libraries)
ติดตั้งไลบรารีที่จำเป็นและตั้งค่าฟอนต์ภาษาไทยเพื่อรองรับการแสดงผลข้อมูล

In [ ]:
import os
import subprocess
import sys

def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Install OpenAI SDK for OpenRouter & Dependencies
# typhoon_ocr is optional (backend-dependent), but pandas/dotenv/tqdm are core.
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 
     'openai', 'pandas', 'tqdm', 'python-dotenv', 'typhoon_ocr'])

Running: /usr/bin/python3 -m pip install -q -U openai pandas tqdm python-dotenv typhoon_ocr
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you 

# 2. Data Loading & Initial Inspection
## 2.1 การตั้งค่าฟอนต์ภาษาไทย
ในส่วนนี้จะทำการตั้งค่าฟอนต์ภาษาไทยเพื่อให้สามารถแสดงผลได้อย่างถูกต้อง

In [2]:
# Auto-detect dataset path to fix FileNotFoundError
import os

possible_roots = [
    '/kaggle/input/super-ai-engineer-season-6-ocr-2569/data',      # Standard Kaggle
    '/kaggle/input/super-ai-engineer-season-6-ocr-2569',           # Alternative Kaggle
    './data/data',                                                 # Local unzip style 1
    './data',                                                      # Local unzip style 2
    '.'                                                            # Current dir
]

TEMPLATE_CSV = None
BASE_DIR = ''

for root in possible_roots:
    path = os.path.join(root, 'submission_template_v3.csv')
    if os.path.exists(path):
        TEMPLATE_CSV = path
        BASE_DIR = root
        break

if TEMPLATE_CSV is None:
    # Fallback default if nothing found
    BASE_DIR = '/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data'
    TEMPLATE_CSV = os.path.join(BASE_DIR, 'submission_template_v3.csv')
    print(f"Warning: Dataset not found. Defaulting to: {TEMPLATE_CSV}")

IMAGE_DIR = os.path.join(BASE_DIR, 'images')
SAMPLE_LABEL_DIR = os.path.join(BASE_DIR, 'sample_labels')

# Working paths
PIPELINE_SCRIPT = 'embedded_pipeline.py'
OUTPUT_CSV = 'submission.csv'
SAMPLE_GOLD_CSV = 'sample_gold.csv'

print(f"Template path: {TEMPLATE_CSV}")
print(f"Image dir:     {IMAGE_DIR}")
print(f"Label dir:     {SAMPLE_LABEL_DIR}")

Template path: /kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/submission_template_v3.csv
Image dir:     /kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/images
Label dir:     /kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/sample_labels


### 2.2 ตรวจสอบโครงสร้างข้อมูล (Inspect Dataset Structure)
ตรวจสอบข้อมูลในไฟล์ CSV เพื่อดูว่ามีคอลัมน์และข้อมูลที่ต้องการครบถ้วนหรือไม่

### 2.3 ตรวจสอบข้อมูลเบื้องต้น (Initial Data Inspection)
แสดงตัวอย่างข้อมูลบางส่วนและตรวจสอบว่ามีค่า missing หรือไม่

In [ ]:
import base64
import json
import re
import threading
import time
import os
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import get_close_matches
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
API_KEY = "" # ใส่ Key
os.environ["OPENROUTER_API_KEY"] = API_KEY
MAX_WORKERS = os.cpu_count() or 4
CACHE_DIR = Path("./cache_sota")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. CORE CLASSES (OOP Style) ---

class RateLimitGuard:
    """Helper to manage API rate limits (Prevent 429 Errors)"""
    def __init__(self, rpm=60):
        self.interval = 60.0 / rpm
        self.last_call = 0
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.monotonic()
            diff = now - self.last_call
            if diff < self.interval:
                time.sleep(self.interval - diff)
            self.last_call = time.monotonic()

class GeminiVisionOCR:
    """
    'The Eyes': Specialized class for reading Thai Election Forms using VLM
    """
    RULES = (
        "This is a page from a Thai official election result form (สส.6/1). "
        "Your task is to perform a complete, verbatim transcription of every piece of information on this page. "
        "Output only Markdown — no commentary, no explanations, nothing outside the transcription.\n\n"
        "Follow these rules strictly:\n"
        "1. **Transcribe every row** of every table without exception. Never skip, truncate, or summarise rows.\n"
        "2. **Preserve exact Thai text** for all party names (ชื่อพรรค), candidate names (ชื่อ-สกุล), "
        "   section labels, and header fields exactly as printed.\n"
        "3. **Convert Thai numerals** (๐๑๒๓๔๕๖๗๘๙) to Arabic digits (0–9) in all numeric fields.\n"
        "4. **For vote-count tables**, reproduce each data row as a Markdown table row with columns:\n"
        "   | หมายเลข | ชื่อ-สกุล | สังกัดพรรคการเมือง | คะแนน |\n"
        "   Include the header row and a separator row.\n"
        "5. **For summary statistics** (numbered items such as 1.1, 1.2, 2.1, 3.1 …), transcribe each item "
        "   on its own line in the format: `<item number> <label> <value>`.\n"
        "6. **Transcribe all header fields** (province, constituency/district name, polling station, date, "
        "   official signatures section labels) verbatim.\n"
        "7. If a cell is blank or illegible, write `-`.\n"
    )

    def __init__(self, api_key: str):
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
        self.model = "google/gemini-2.0-flash-001"
        self.guard = RateLimitGuard(rpm=50)

    def process_image(self, image_path: Path) -> str:
        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        
        self.guard.wait()
        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                        {"type": "text", "text": self.RULES},
                    ],
                }],
                temperature=0.0
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            print(f"Vision Error: {e}")
            return ""

class VoteLogicParser:
    """
    'The Brain': Extracts structured JSON votes from raw OCR text
    """
    CONSTITUENCY_PROMPT = """
    Task: Extract vote counts for these specific parties from the provided constituency election text.
    Input Text: "{text}"
    Target Parties: {parties}
    Output: JSON format {{"Party Name": VoteCount (int)}}. Use 0 if missing.
    """
    
    PARTY_LIST_PROMPT = """
    Task: Extract vote counts for these specific parties from the provided party-list election text.
    Input Text: "{text}"
    Target Parties: {parties}
    Output: JSON format {{"Party Name": VoteCount (int)}}. Use 0 if missing.
    """

    def __init__(self, api_key: str):
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
        self.model = "google/gemini-2.0-flash-001"
        self.guard = RateLimitGuard(rpm=50)

    def extract(self, doc_id: str, raw_text: str, parties: list) -> dict:
        self.guard.wait()
        is_party_list = "party_list" in doc_id
        template = self.PARTY_LIST_PROMPT if is_party_list else self.CONSTITUENCY_PROMPT
        
        prompt = template.format(text=raw_text[:12000], parties="\n".join(parties))
        
        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.0
            )
            return json.loads(resp.choices[0].message.content)
        except:
            return {}

print("Pipeline Initialized: Engines Ready.")

Pipeline Initialized: Engines Ready.


# 3. OCR Pipeline Setup
### 3.1 ตั้งค่า Pipeline สำหรับ OCR
กำหนดค่า backend และ rate limiter สำหรับการประมวลผล OCR

In [4]:
# --- 3. DATASET PREPARATION ---

class DatasetManager:
    """Helper class to load images and classify document types."""
    def __init__(self, img_dir: Path, template_csv: Path):
        self.img_dir = img_dir
        self.doc_map: Dict[str, List[Path]] = defaultdict(list)
        self.party_template: Dict[str, List[str]] = defaultdict(list)
        
        # Load Images
        print(f"Loading Images from {img_dir}...")
        for p in self.img_dir.glob("*.png"):
            self._load_file(p)
            
        # Sort Pages
        for d_id in self.doc_map:
            self.doc_map[d_id].sort(key=self._page_sorter)
            
        # Load Template Targets
        print(f"Loading Template from {template_csv}...")
        df = pd.read_csv(template_csv, dtype=str)
        if 'party_name' in df.columns:
            for d_id, group in df.groupby("doc_id"):
                self.party_template[d_id] = group["party_name"].dropna().unique().tolist()
                
    def _load_file(self, path: Path):
        # constituency_10_1_page2.png -> constituency_10_1
        match = re.search(r"(.+?)(_page(\d+))?\.png$", path.name)
        if match:
            doc_id = match.group(1)
            self.doc_map[doc_id].append(path)
            
    def _page_sorter(self, path: Path) -> int:
        match = re.search(r"_page(\d+)", path.name)
        return int(match.group(1)) if match else 1

    def get_groups(self):
        return self.doc_map

dataset = DatasetManager(Path(IMAGE_DIR), Path(TEMPLATE_CSV))
print(f"Found {len(dataset.get_groups())} document groups.")

Loading Images from /kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/images...
Loading Template from /kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/submission_template_v3.csv...
Found 300 document groups.


# 4. Batch OCR Processing
### 4.1 การประมวลผล OCR แบบกลุ่ม
ใช้ Pipeline ที่ตั้งค่าไว้เพื่อประมวลผล OCR สำหรับเอกสารทั้งหมด

In [8]:
# --- 4. BATCH PIPELINE (OOP Style) ---

class ElectionPipeline:
    """
    Main Pipeline Controller:
    - Loads Documents
    - Runs Vision Model
    - Extract Votes via LLM
    - Maps to Submission Template
    """
    def __init__(self, api_key: str):
        self.vision_engine = GeminiVisionOCR(api_key)
        self.parser_engine = VoteLogicParser(api_key)
        # Change cache to store pages individually: {doc_id: {page_num: text}}
        self.cache: Dict[str, Dict[int, str]] = {} 
        self.final_votes: Dict[str, Dict[str, int]] = {}
        
    def run_vision_batch(self, doc_groups: Dict[str, List[Path]]):
        """Scan all images in parallel"""
        print("Starting Vision Batch...")
        
        tasks_todo = []
        for d_id, pages_list in doc_groups.items():
            if d_id not in self.cache:
                self.cache[d_id] = {}
            
            for i, p_path in enumerate(pages_list):
                pg_num = i + 1
                if pg_num not in self.cache[d_id]:
                    tasks_todo.append((d_id, pg_num, p_path))
        
        print(f"Tasks: {len(tasks_todo)}")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            # Future key maps to metadata (d_id, pg_num)
            futures = {pool.submit(self.vision_engine.process_image, p): (d, pg) for d, pg, p in tasks_todo}
            
            with tqdm(total=len(tasks_todo), desc="Scanning") as pbar:
                for future in as_completed(futures):
                    d_id, pg_num = futures[future] # Retrieve metadata from dict
                    try:
                        text = future.result() # Retrieve result from future
                        if text:
                            self.cache[d_id][pg_num] = text
                    except Exception as e:
                        print(f"Error on {d_id} p{pg_num}: {e}")
                    pbar.update(1)
                    
    def run_extraction_batch(self, doc_parties_map: Dict[str, List[str]]):
        """Convert Texts to Key-Value Votes"""
        print("Starting Extraction Batch...")
        
        tasks = []
        for d_id, pages_dict in self.cache.items():
            if not pages_dict: 
                continue
                
            # Combine pages in order
            sorted_pages = sorted(pages_dict.keys())
            raw_txt = "\n\n".join([pages_dict[p] for p in sorted_pages])
            
            if d_id in doc_parties_map and d_id not in self.final_votes:
                tasks.append((d_id, raw_txt, doc_parties_map[d_id]))
                
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(self.parser_engine.extract, d, t, p): d for d, t, p in tasks}
            
            with tqdm(total=len(tasks), desc="Parsing") as pbar:
                for f in as_completed(futures):
                    d_id_key = futures[f]
                    try:
                        result = f.result() # extract returns a dict of votes
                        if isinstance(result, dict):
                            self.final_votes[d_id_key] = result
                        else:
                            print(f"Warning: {d_id_key} returned non-dict result ({type(result)}). Skipping.")
                    except Exception as e:
                        print(f"Error parsing {d_id_key}: {e}")
                    pbar.update(1)
                        
    def get_results(self):
        return self.final_votes

# --- EXECUTION ---
# Ensure Dataset Manager is ready from previous cell
if 'dataset' not in globals():
    print("Dataset Manager not found. Please run previous cell.")
else:
    pipeline = ElectionPipeline(API_KEY)
    pipeline.run_vision_batch(dataset.get_groups())
    pipeline.run_extraction_batch(dataset.party_template)

    print("Pipeline Finished.")

Starting Vision Batch...
Tasks: 846


Scanning:   0%|          | 0/846 [00:00<?, ?it/s]

Starting Extraction Batch...


Parsing:   0%|          | 0/300 [00:00<?, ?it/s]

Pipeline Finished.


# 5. การสร้างไฟล์ Submission
### 5.1 การเตรียมข้อมูลสำหรับการส่งผลลัพธ์
ในส่วนนี้จะทำการสร้างไฟล์ submission.csv โดยใช้ข้อมูลที่ได้จากการประมวลผล OCR

In [10]:
# --- 5. GENERATE SUBMISSION ---
# Convert results into CSV Format

final_data = pipeline.get_results()
sample_df = pd.read_csv(TEMPLATE_CSV, dtype=str)

output_rows = []
for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Saving"):
    doc_id = row['doc_id']
    party = row.get('party_name', '')
    
    votes = 0
    if doc_id in final_data:
        doc_data = final_data[doc_id]
        
        # Guard against corrupted data structure (e.g. list instead of dict)
        if isinstance(doc_data, dict):
            # Check Exact Match
            if party in doc_data:
                votes = doc_data[party]
            else:
                # Check Fuzzy Match
                if isinstance(party, str):  # Ensure party is a string
                    matches = get_close_matches(party, doc_data.keys(), n=1, cutoff=0.7)
                    if matches:
                        votes = doc_data[matches[0]]
                else:
                    print(f"Warning: Party name is not a string (got {type(party)}). Skipping fuzzy match.")
        else:
            # If doc_data is not a dict, skip and warn (already logged in batch step)
            pass
                
    output_rows.append({"id": row['id'], "votes": votes})

pd.DataFrame(output_rows).to_csv(OUTPUT_CSV, index=False)
print(f"File Saved: {OUTPUT_CSV}")
pd.DataFrame(output_rows).head()

Saving:   0%|          | 0/10053 [00:00<?, ?it/s]

File Saved: submission.csv


,id,votes
0,constituency_10_1_1,0
1,constituency_10_1_2,0
2,constituency_10_1_3,0
3,constituency_10_1_4,0
4,constituency_10_1_5,0
